# TACET v2 Tier 4 Wave B — Thin Colab Runner

Mounts Google Drive for persistent atomic checkpoints (falls back to `/content/tacet_wave_b` outside Colab), installs `n24q02m/tacet`, and executes `experiments/run_tier4_qlora.py` across Arms A/B/C with `max_seq_len=2560` and `max_new_tokens=1536`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

TACET_REF = os.environ.get("TACET_REF", "main")
os.environ.setdefault("HF_HOME", "/content/hf_cache")

output_dir = Path("/content/tacet_wave_b")
try:
    from google.colab import drive

    drive.mount("/content/drive")
    output_dir = Path("/content/drive/MyDrive/tacet_wave_b")
except Exception:
    pass
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Checkpoint output_dir={output_dir}")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"tacet[experiments] @ git+https://github.com/n24q02m/tacet.git@{TACET_REF}",
        "peft",
        "bitsandbytes",
        "transformers",
        "accelerate",
    ]
)

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

repo_dir = Path("/content/tacet")
if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.check_call(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        TACET_REF,
        "https://github.com/n24q02m/tacet.git",
        str(repo_dir),
    ]
)
subprocess.check_call(
    [
        sys.executable,
        str(repo_dir / "experiments" / "run_tier4_qlora.py"),
        "--arms",
        "A",
        "B",
        "C",
        "--seeds",
        "0",
        "1",
        "2",
        "--model-id",
        "Qwen/Qwen3-1.7B",
        "--model-revision",
        "70d244cc86cc",
        "--max-seq-len",
        "2560",
        "--max-new-tokens",
        "1536",
        "--output-dir",
        str(output_dir),
    ]
)